In [0]:
# ===============================================
# Notebook: KPI - Lowest Brand by Region
# ===============================================

from pyspark.sql.functions import col, row_number, sum as _sum, round
from pyspark.sql.window import Window

catalog = "beverage_analytics"
schema = "delivery"
table = "kpi_lowest_brand_by_region"

# --------------------------------------
# 1. Leitura das tabelas
# --------------------------------------
df_fact_sales = spark.table(f"{catalog}.fact.fact_sales")

# --------------------------------------
# 2. Query KPI
# --------------------------------------
kpi_df = df_fact_sales \
    .groupBy("brand_nm", "btlr_org_lvl_c_desc") \
    .agg(round(_sum("volume"), 2).alias("total_sales"))

windowSpec = Window.partitionBy("btlr_org_lvl_c_desc").orderBy(col("total_sales").asc())
kpi_ranked = kpi_df.withColumn("rank", row_number().over(windowSpec)).filter(col("rank") == 1)

# --------------------------------------
# 3. Gravar resultado como Delta Table
# --------------------------------------
kpi_ranked.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable(f"{catalog}.{schema}.{table}")

print(f"KPI gravado com sucesso em: {catalog}.{schema}.{table}")
